# Alokasi Uang Anak Kos: Forward Chaining vs Fuzzy Sugeno

Notebook untuk menguji dan mencocokkan hitungan dengan papan tulis.

**Input otomatis:** `jatah = saldo / max(sisa_hari, 1)` lalu `r = jatah / B` (B = biaya primer harian: makan + transport).

**Cara pakai:** jalankan sel dari atas ke bawah (Kernel → Restart & Run All). Kalau ada file `aturan.json` di folder yang sama, notebook memakainya; kalau tidak, memakai aturan bawaan.

> Semua angka aturan adalah **contoh** sampai ada data survei.

## 1. Inti program (aturan, fuzzifikasi, forward chaining, Sugeno)

In [ ]:
import copy, json
from itertools import product
from pathlib import Path

INF = float("inf")

# (kategori jatah, kategori sisa hari) -> (Primer, Sekunder, Tersier), total 100
_TABEL = {
    ("sedikit", "dekat"): (80, 15, 5),
    ("sedikit", "sedang"): (85, 12, 3),
    ("sedikit", "jauh"): (90, 10, 0),
    ("sedang", "dekat"): (60, 25, 15),
    ("sedang", "sedang"): (65, 23, 12),
    ("sedang", "jauh"): (70, 22, 8),
    ("banyak", "dekat"): (45, 30, 25),
    ("banyak", "sedang"): (50, 30, 20),
    ("banyak", "jauh"): (55, 28, 17),
}

DEFAULT_CONFIG = {
    "_catatan": "Semua angka adalah CONTOH. Ganti dengan hasil survei/jurnal.",
    "B": 30000,  # biaya primer harian (makan + transport), Rp
    "kategori": ["primer", "sekunder", "tersier"],
    # Himpunan trapesium [a, b, c, d]; null = tak terhingga (bahu kiri/kanan)
    "input": {
        "jatah": {  # r = jatah per hari / B
            "sedikit": [None, None, 1, 1.5],
            "sedang": [1, 1.5, 2, 2.5],
            "banyak": [2, 2.5, None, None],
        },
        "hari": {  # sisa hari sampai gajian
            "dekat": [None, None, 5, 10],
            "sedang": [5, 10, 15, 20],
            "jauh": [15, 20, None, None],
        },
    },
    "aturan": [
        {"jika": {"jatah": j, "hari": h}, "maka": list(z)} for (j, h), z in _TABEL.items()
    ],
}

PESAN_PERINGATAN = "Dana tidak cukup untuk kebutuhan primer"
VARIABEL = ("jatah", "hari")


def muat_config(path=None):
    """path=None -> config bawaan. Selain itu baca JSON lalu validasi."""
    if path is None:
        return copy.deepcopy(DEFAULT_CONFIG)
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _params(lst):
    a, b, c, d = lst
    a = -INF if a is None else float(a)
    b = -INF if b is None else float(b)
    c = INF if c is None else float(c)
    d = INF if d is None else float(d)
    if not (a <= b <= c <= d):
        raise ValueError("Titik trapesium harus a<=b<=c<=d: %s" % (lst,))
    return a, b, c, d


def validasi(cfg):
    """Raise ValueError jika config salah; return daftar peringatan (boleh kosong)."""
    peringatan = []
    n = len(cfg["kategori"])
    if cfg["B"] <= 0:
        raise ValueError("B harus > 0")
    for var in VARIABEL:
        if var not in cfg["input"] or not cfg["input"][var]:
            raise ValueError("Variabel input '%s' belum didefinisikan" % var)
        for nama, p in cfg["input"][var].items():
            if len(p) != 4:
                raise ValueError("Himpunan %s.%s harus punya 4 titik" % (var, nama))
            _params(p)
    for i, a in enumerate(cfg["aturan"], 1):
        if len(a["maka"]) != n:
            raise ValueError("Aturan R%d: jumlah konsekuen harus %d" % (i, n))
        if abs(sum(a["maka"]) - 100) > 1e-9:
            raise ValueError("Aturan R%d: total %s, harus 100" % (i, sum(a["maka"])))
        for var, s in a["jika"].items():
            if s not in cfg["input"].get(var, {}):
                raise ValueError("Aturan R%d: himpunan '%s' tidak ada di '%s'" % (i, s, var))
    ada = {tuple(a["jika"].get(v) for v in VARIABEL) for a in cfg["aturan"]}
    for kombinasi in product(*(cfg["input"][v] for v in VARIABEL)):
        if kombinasi not in ada:
            peringatan.append("Kombinasi %s belum punya aturan" % (kombinasi,))
    return peringatan


def trapezoid(x, a, b, c, d):
    if b <= x <= c:
        return 1.0
    if x < b:
        return 0.0 if x <= a else (x - a) / (b - a)
    return 0.0 if x >= d else (d - x) / (d - c)


def nilai_input(cfg, saldo, sisa_hari):
    jatah = saldo / max(sisa_hari, 1)  # max(...,1): hindari bagi nol di hari gajian
    return jatah, {"jatah": jatah / cfg["B"], "hari": float(sisa_hari)}


def derajat(cfg, crisp):
    return {
        var: {nama: trapezoid(crisp[var], *_params(p)) for nama, p in cfg["input"][var].items()}
        for var in crisp
    }


def _peringatan(cfg, jatah):
    return PESAN_PERINGATAN if jatah < cfg["B"] else None


def sugeno(cfg, saldo, sisa_hari):
    jatah, crisp = nilai_input(cfg, saldo, sisa_hari)
    mu = derajat(cfg, crisp)
    n = len(cfg["kategori"])
    pembilang, penyebut, aktif = [0.0] * n, 0.0, []
    for i, a in enumerate(cfg["aturan"], 1):
        alpha = min(mu[v][s] for v, s in a["jika"].items())  # AND = min
        if alpha > 0:
            aktif.append({"aturan": "R%d" % i, "jika": a["jika"], "alpha": alpha, "z": a["maka"]})
            penyebut += alpha
            for k in range(n):
                pembilang[k] += alpha * a["maka"][k]
    if penyebut == 0:
        raise ValueError("Tidak ada aturan aktif: cek himpunan dan aturan di config")
    return {
        "jatah": jatah, "r": crisp["jatah"], "derajat": mu, "aktif": aktif,
        "sum_alpha": penyebut, "persen": [p / penyebut for p in pembilang],
        "peringatan": _peringatan(cfg, jatah),
    }


def forward_chaining(cfg, saldo, sisa_hari):
    """Klasifikasi tegas (himpunan dengan derajat tertinggi; seri -> yang pertama
    di config) -> fakta -> aturan IF-THEN dipicu berulang sampai tidak ada fakta baru."""
    jatah, crisp = nilai_input(cfg, saldo, sisa_hari)
    mu = derajat(cfg, crisp)
    fakta = set()
    for var, himpunan in mu.items():
        if max(himpunan.values()) == 0:
            raise ValueError("Nilai %s di luar semua himpunan" % var)
        fakta.add("%s=%s" % (var, max(himpunan, key=himpunan.get)))
    jejak, hasil, berubah = [], None, True
    while berubah:
        berubah = False
        for i, a in enumerate(cfg["aturan"], 1):
            syarat = {"%s=%s" % (v, s) for v, s in a["jika"].items()}
            kesimpulan = "alokasi=" + "/".join(str(x) for x in a["maka"])
            if syarat <= fakta and kesimpulan not in fakta:
                fakta.add(kesimpulan)
                jejak.append("R%d: IF %s THEN %s" % (i, " AND ".join(sorted(syarat)), kesimpulan))
                if hasil is None:
                    hasil = a["maka"]
                berubah = True
    if hasil is None:
        raise ValueError("Tidak ada aturan yang cocok dengan fakta: %s" % sorted(fakta))
    return {
        "jatah": jatah, "r": crisp["jatah"], "fakta": sorted(fakta), "jejak": jejak,
        "persen": [float(x) for x in hasil], "peringatan": _peringatan(cfg, jatah),
    }


def rp(x):
    return "Rp" + "{:,.0f}".format(x).replace(",", ".")


def p1(persen):
    """Bulatkan 1 desimal, .x5 ke atas (sama dengan hitungan manual)."""
    return [round(x + 1e-9, 1) for x in persen]

## 2. Pilih konfigurasi
Ubah angka di `cfg` langsung di sel ini untuk bereksperimen (misal `cfg["B"] = 40000`).

In [ ]:
if Path("aturan.json").exists():
    cfg = muat_config("aturan.json")
    print("Memakai aturan.json")
else:
    cfg = muat_config()
    print("Memakai konfigurasi bawaan (contoh)")

print("Peringatan validasi:", validasi(cfg) or "tidak ada")
print("B =", rp(cfg["B"]))

## 3. Langkah demi langkah (format papan tulis)

In [ ]:
def tampilkan_langkah(cfg, saldo, hari):
    s = sugeno(cfg, saldo, hari)
    print(f"Saldo {rp(saldo)} | sisa {hari} hari | B {rp(cfg['B'])}")
    print(f"jatah = {rp(saldo)} / {max(hari, 1)} = {rp(s['jatah'])}  ->  r = jatah/B = {s['r']:.3f}\n")

    print("LANGKAH 1 - Fuzzifikasi (hanya derajat > 0)")
    for var, himp in s["derajat"].items():
        nilai = ", ".join(f"{n} = {m:.3f}" for n, m in himp.items() if m > 0)
        print(f"  {var:<6}: {nilai}")

    print("\nLANGKAH 2 - alpha = min(...) tiap aturan aktif")
    for a in s["aktif"]:
        syarat = " & ".join(f"{v}={x}" for v, x in a["jika"].items())
        print(f"  {a['aturan']} [{syarat}]  alpha = {a['alpha']:.3f}  z = {a['z']}")
    print(f"  Sum alpha = {s['sum_alpha']:.3f}")

    print("\nLANGKAH 3 - rata-rata terbobot  z = sum(alpha*z) / sum(alpha)")
    for k, nama in enumerate(cfg["kategori"]):
        suku = " + ".join(f"{a['alpha']:.3f}*{a['z'][k]}" for a in s["aktif"])
        print(f"  {nama:<9}= ({suku}) / {s['sum_alpha']:.3f} = {p1(s['persen'])[k]}%")

    print("\nRupiah per hari:")
    for nama, v in zip(cfg["kategori"], s["persen"]):
        print(f"  {nama:<9} {rp(s['jatah'] * v / 100)}")
    if s["peringatan"]:
        print("\nPERINGATAN:", s["peringatan"])
    return s

_ = tampilkan_langkah(cfg, 500_000, 8)

## 4. Bandingkan Forward Chaining dan Sugeno
Perhatikan baris jatah Rp37.000 vs Rp38.000: Forward Chaining melompat, Sugeno bergeser halus.

In [ ]:
kasus = [(37_000 * 12, 12), (38_000 * 12, 12), (500_000, 8), (450_000, 12), (300_000, 4), (300_000, 20)]
print(f"{'saldo':>10} {'hari':>4} {'jatah/hr':>9} | {'Forward Chaining P/S/T':<24} | {'Sugeno P/S/T'}")
for saldo, hari in kasus:
    f, s = forward_chaining(cfg, saldo, hari), sugeno(cfg, saldo, hari)
    fc = " / ".join(f"{x:.1f}" for x in p1(f["persen"]))
    su = " / ".join(f"{x:.1f}" for x in p1(s["persen"]))
    print(f"{saldo:>10,} {hari:>4} {s['jatah']:>9,.0f} | {fc:<24} | {su}")

## 5. Tes otomatis
Jika semua `assert` lulus, tidak ada output error.

In [ ]:
def dekat(a, b, tol=0.06):
    return all(abs(x - y) <= tol for x, y in zip(a, b))

base = muat_config()  # tes selalu memakai konfigurasi bawaan
s = sugeno(base, 500_000, 8)
assert p1(s["persen"]) == [59.1, 25.4, 15.5] and abs(s["sum_alpha"] - 4 / 3) < 1e-9
assert dekat(sugeno(base, 300_000, 4)["persen"], (45, 30, 25))
assert dekat(sugeno(base, 300_000, 20)["persen"], (90, 10, 0))
assert sugeno(base, 300_000, 20)["peringatan"] and not sugeno(base, 300_000, 4)["peringatan"]
assert dekat(sugeno(base, 450_000, 12)["persen"], (75, 17.5, 7.5))
assert forward_chaining(base, 37_000 * 12, 12)["persen"] == [85.0, 12.0, 3.0]
assert forward_chaining(base, 38_000 * 12, 12)["persen"] == [65.0, 23.0, 12.0]
assert dekat(sugeno(base, 37_000 * 12, 12)["persen"], (75.7, 17.1, 7.2))
assert abs(sum(sugeno(base, 100_000, 0)["persen"]) - 100) < 1e-9   # hari gajian
print("Semua tes lulus.")

## 6. Grafik

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib belum terpasang: pip install matplotlib")

if plt:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.4))
    xs = [i / 100 for i in range(0, 351)]
    for nama, p in cfg["input"]["jatah"].items():
        ax[0].plot(xs, [trapezoid(x, *_params(p)) for x in xs], label=nama)
    ax[0].set(title="Himpunan jatah (r = jatah/B)", xlabel="r", ylabel="derajat")
    ax[0].legend()
    hs = list(range(0, 31))
    for nama, p in cfg["input"]["hari"].items():
        ax[1].plot(hs, [trapezoid(h, *_params(p)) for h in hs], label=nama)
    ax[1].set(title="Himpunan sisa hari", xlabel="hari sampai gajian", ylabel="derajat")
    ax[1].legend()
    plt.tight_layout()
    plt.show()

In [ ]:
if plt:
    hari = 12
    jatah_list = list(range(10_000, 100_001, 500))
    fc_p = [forward_chaining(cfg, j * hari, hari)["persen"][0] for j in jatah_list]
    su_p = [sugeno(cfg, j * hari, hari)["persen"][0] for j in jatah_list]
    plt.figure(figsize=(7.5, 3.8))
    plt.plot([j / 1000 for j in jatah_list], fc_p, label="Forward Chaining (melompat)")
    plt.plot([j / 1000 for j in jatah_list], su_p, label="Fuzzy Sugeno (mulus)")
    plt.xlabel("jatah per hari (ribu Rp)")
    plt.ylabel("porsi primer (%)")
    plt.title(f"Porsi primer vs jatah per hari (sisa {hari} hari)")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

## 7. Coba sendiri (slider)
Butuh `ipywidgets` (`pip install ipywidgets`). Kalau belum ada, ubah nilai di sel berikutnya secara manual.

In [ ]:
try:
    import ipywidgets as w

    def coba(saldo=500_000, hari=8, B=cfg["B"]):
        c = copy.deepcopy(cfg)
        c["B"] = B
        try:
            tampilkan_langkah(c, saldo, hari)
        except ValueError as e:
            print("Tidak bisa dihitung:", e)

    w.interact(
        coba,
        saldo=w.IntSlider(value=500_000, min=50_000, max=2_000_000, step=10_000, description="saldo"),
        hari=w.IntSlider(value=8, min=0, max=40, description="sisa hari"),
        B=w.IntSlider(value=cfg["B"], min=10_000, max=60_000, step=1_000, description="B"),
    )
except ImportError:
    print("ipywidgets belum terpasang. Pakai cara manual di bawah.")
    _ = tampilkan_langkah(cfg, 500_000, 8)